# Summary Report : Mixed-Signal Analyzer (Finance & NLP)

## 1. Introduction and Background

The main objective of this project is to design and deploy an end-to-end Machine Learning pipeline capable of predicting the short-term trend of a financial asset. Specifically, the system must determine, through binary classification, whether the price of the target stock (in this case, Apple APPL) will rise or fall over a 5-day horizon.

The uniqueness and added value of this project lie in the integration of data. To best mimic a trader’s analysis, the model combines two distinct sources of information :
* Quantitative signals : Traditional time-series analysis, including price history and the calculation of technical indicators.
* Qualitative signals : Natural Language Processing (NLP) applied daily to financial news to capture market psychology and “sentiment.” 

Beyond pure prediction, the architecture of this project was designed to demonstrate complete mastery of the data value chain, including :
* Data Engineering : Creation of an automated ETL pipeline (Extraction via API, Transformation, and Loading).
* Statistical Modeling : Comparing advanced Machine Learning algorithms to solve a complex classification problem.
* Software Development : Structuring code into modular Python scripts, database management (SQL), and applying best practices for version control (Git).

```mermaid
graph TD
    %% Sources de données
    API1[(Yahoo Finance)] -->|Prix Historiques| EXT(Extract)
    API2[(NewsAPI)] -->|Articles Financiers| EXT
    
    %% Extraction et Transformation
    EXT --> TR(Transform)
    TR -->|Modèle FinBERT| NLP[Analyse de Sentiment]
    TR -->|Mathématiques| FE[Indicateurs Techniques]
    
    %% Stockage SQL
    NLP --> SQL[(Load : Base de données SQLite)]
    FE --> SQL
    
    %% Machine Learning
    SQL -->|Données Hybrides| ML(Machine Learning)
    ML -->|Entraînement| XGB[XGBoost vs Random Forest]
    XGB --> P((Prédiction : Hausse / Baisse))
    
    %% Styles personnalisés
    style API1 fill:#0ea5e9,stroke:#0369a1,stroke-width:2px,color:#ffffff
    style API2 fill:#0ea5e9,stroke:#0369a1,stroke-width:2px,color:#ffffff
    style EXT fill:#f1f5f9,stroke:#64748b,stroke-width:1px,color:#0f172a
    style TR fill:#f1f5f9,stroke:#64748b,stroke-width:1px,color:#0f172a
    style NLP fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b
    style FE fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b
    style SQL fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f
    style ML fill:#f1f5f9,stroke:#64748b,stroke-width:1px,color:#0f172a
    style XGB fill:#ddd6fe,stroke:#7c3aed,stroke-width:2px,color:#2e1065
    style P fill:#10b981,stroke:#047857,stroke-width:2px,color:#ffffff
```

## 2. Feature Engineering and Signal Justification

A model’s performance depends heavily on the quality of the explanatory variables it is fed. For this pipeline, we designed a hybrid dataset that captures both the intrinsic price dynamics (technical analysis) and exogenous market sentiment (sentiment analysis).

### 2.1. Quantitative Signals : Price Dynamics and Risk

* Moving Averages (SMA_20 and EMA_20) : The 20-day simple moving average (SMA) provides a baseline for the short- to medium-term trend. The exponential moving average (EMA), by placing greater weight on recent prices, allows the model to detect trend breaks more quickly.
* Historical Volatility (14 days) : This indicator measures market uncertainty. It is calculated using the moving standard deviation of daily returns, annualized according to the formula: $Volatility = \sigma_{14} \times \sqrt{252}$. Volatility spikes often precede major market reversals, a crucial signal for decision trees.
* Relative Strength Index (RSI_14) : This is a momentum oscillator that measures the speed of price movements to identify overbought or oversold conditions. Its mathematical formula is: $RSI = 100 - \frac{100}{1 + RS}$, where $RS$ (Relative Strength) represents the ratio of the average gains to the average losses over 14 days.

### 2.2. Qualitative Signals : NLP and Market Psychology

Pure quantitative models suffer from an inherent lag because they respond only to past prices. The integration of textual data aims to give the model the ability to anticipate based on the flow of information.

* The Choice of FinBERT : General-purpose NLP models often fail to capture the nuances of financial jargon (for example, the word “drop” can be positive when referring to the unemployment rate). We implemented FinBERT, a Transformer-based model specifically retrained by ProsusAI on a massive financial corpus (Financial PhraseBank).
* Daily Aggregation (daily_sentiment and new_volumes) : FinBERT’s raw predictions (probabilities for the Positive, Negative, and Neutral classes) were weighted and aggregated on a daily basis. This allows us to transform an unstructured news feed into a continuous time series that aligns perfectly with our market data in the SQL table `fact_news_sentiment`.

Below is a chart showing the performance of Apple stock, illustrating the crossover of moving averages and the overbought/oversold zones identified by the RSI

![Analyse Technique AAPL](../Images/SMA_Close_.png)

This chart shows the daily values of the “daily_sentiment” index as well as the number of news stories about Apple

![Analyse Technique AAPL](../Images/daily_sentiment.png)

## 3. Model Evaluation and Performance

### 3.1. Evaluation Methodology and Temporal Validation

In financial modeling, the use of traditional cross-validation (such as random K-fold cross-validation) should be avoided, as it would result in data leakage from the future to the past.

To ensure the integrity of our tests on Apple stock, we have implemented several tools:
* A strict chronological split (80% / 20%) : The model is trained solely on the distant past and evaluated exclusively on the most recent data.
* A TimeSeriesSplit (temporal cross-validation) : Hyperparameter optimization via GridSearchCV adhered to the chronological order of the training blocks to simulate a real trading environment under actual conditions.
* A binary classification formulation : The target variable indicates whether the closing price at a 5-day horizon will be higher ($1$) or lower ($0$) than the current price.

### 3.2. Comparative Analysis: Random Forest vs. XGBoost

The two ensemble algorithms tested revealed radically different learning dynamics on our dataset

* The Behavior of the Random Forest (Conservative Model) : The Random Forest builds independent trees in parallel. Faced with the inherent noise in stock market time series, the average of the forest’s votes adopted an extreme smoothing strategy. On the test set, the model favored the majority class, demonstrating an inability to isolate weak signals of trend reversals. Although it seeks to counter overfitting, its independent structure proved too rigid to capture the nonlinearity of mixed signals (price + sentiment).
* The behavior of XGBoost (Sequential Gradient Boosting) : In contrast, XGBoost builds its trees iteratively; each new tree is specifically trained to correct the residual errors of the previous trees. This sequential approach allowed it to adapt precisely to price fluctuations and outperform, achieving an overall accuracy higher than that of Random Forest.

### 3.3.  Analysis of Advanced Metrics (ROC, Precision-Recall and Confusion Matrix)

To validate the robustness of XGBoost, the analysis goes beyond overall accuracy :
* ROC and Precision-Recall Curves : Analysis of probability scores using the area under the curve (AUC) confirms the classifier’s robustness, demonstrating sufficient discriminatory power to support the automation of decision orders.
* The Confusion Matrix : It highlights the model’s ability to correctly identify uptrend and downtrend zones, minimizing false signals compared to the Random Forest.

## 4. Feature Importance and Final Conclusion

### 4.1. Feature Importance Analysis

To interpret the XGBoost model's decision-making process, we extracted the Feature Importance metric (measured by Information Gain). This allows us to quantify the exact contribution of each variable in reducing prediction uncertainty and identifying the strongest market signals.

Based on the model's algorithmic splits, we can observe a clear hierarchy in the data :

* Price Dynamics : Quantitative indicators such as Volatility_14 and RSI_14 typically form the foundational pillars of the model. Historical price action continues to capture the bulk of the market's mechanical mean-reversion and momentum traits.
* The Weight of NLP (daily_sentiment and news_volume) : The sentiment score extracted via FinBERT establishes itself as a highly significant feature. The fact that the algorithm actively selects the sentiment variable for its node splits demonstrates that qualitative data is essential for refining the final prediction.

### 4.2. The Added Value of NLP: Does Sentiment Improve Predictions?

Addressing the core hypothesis of this pipeline : Does news sentiment actually improve predictions compared to a strictly price-based model?  

The empirical evidence from our model evaluation points to Yes. While a purely quantitative model is inherently lagging—reacting only after a price movement has already occurred—the integration of FinBERT provides a pseudo-leading indicator. The algorithmic behavior reveals that during periods of market uncertainty, the daily_sentiment feature acts as a critical context layer. It prevents the model from generating false positive signals by contextualizing mathematical price drops or surges with real-world news polarity (e.g., distinguishing between a technical dip and a structural crisis).

## Conclusion and Production Outlook

This project successfully validates the architecture of a fully automated, hybrid machine learning pipeline. By seamlessly orchestrating data extraction, NLP transformations, SQL relational storage, and predictive modeling, we have built a robust, end-to-end analytical engine capable of processing mixed signals.

Next steps for production scaling:To transition this pipeline into a live trading or asset management environment, the following infrastructure enhancements are recommended :
* Data Scaling: Expanding the historical database across multiple years to expose the XGBoost model to various macroeconomic cycles (bull, bear, and stagnant markets).
* Feature Expansion: Incorporating higher-frequency data and macroeconomic indicators (e.g., interest rates, sector-wide news) to further enrich the predictive context.